In [ ]:
import requests
import os
import re
import time
from dotenv import load_dotenv

import pandas as pd
import matplotlib.pyplot as plt

EuroPMC which containes PMC, PubMed and others.

hits - 287986

In [ ]:
params = {
    "query": "(TITLE_ABS:polymer* OR TITLE_ABS:copolymer*) AND HAS_ABSTRACT:y AND (IN_PMC:y OR IN_EPMC:y OR HAS_PDF:y)",
    "format": "json",
    "resultType": "core",
    "pageSize": 1000,
}

r = requests.get(
    "https://www.ebi.ac.uk/europepmc/webservices/rest/search",
    params=params,
)

results = r.json()["resultList"]["result"]

In [ ]:
results[0]

In [ ]:
r.json()["hitCount"]

In [ ]:
records = []

for item in results:

    journal_info = item.get("journalInfo") or {}
    journal = journal_info.get("journal") or {}

    records.append(
        {
            "doi": item.get("doi", None),
            "title": item.get("title", None),
            "pmcid": item.get("pmcid", None),
            "pmid": item.get("pmid", None),
            "abstract": item.get("abstractText", None),
            "authors": item.get("authorString", None),
            "authorList": item.get("authorList", None),
            "affiliation": item.get("affiliation", None),
            "journal": journal.get("title", None),
            "publication_year": item.get("pubYear", None),
            "is_open_access": (
                item.get("isOpenAccess") == "Y"
                if "isOpenAccess" in item
                else None
            ),
            "in_epmc": (
                item.get("inEPMC") == "Y"
                if "inEPMC" in item
                else None
            ),
            "in_pmc": (
                item.get("inPMC") == "Y"
                if "inPMC" in item
                else None
            ),
            "has_pdf": (
                item.get("hasPDF") == "Y"
                if "hasPDF" in item
                else None
            ),
        }
    )

df = pd.DataFrame(records)

In [ ]:
output_path = os.path.join("..", "data", f"EuroPMC_papers.csv")

# df.to_csv(
#     output_path,
#     sep="\t",
#     index=False,
#     encoding="utf-8",
# )

arXiv hits - 15225

In [ ]:
import xml.etree.ElementTree as ET

search_query = (
    "(ti:polymer OR abs:polymer OR "
    "ti:polymers OR abs:polymers OR "
    "ti:copolymer OR abs:copolymer OR "
    "ti:copolymers OR abs:copolymers OR "
    "ti:polymerization OR abs:polymerization OR "
    "ti:copolymerization OR abs:copolymerization OR "
)

url = "https://export.arxiv.org/api/query"

params = {
    "search_query": search_query,
    "start": 0,
    "max_results": 1000,
    "sortBy": "submittedDate",
    "sortOrder": "descending",
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

root = ET.fromstring(response.text)

In [ ]:
ns = {
    "atom": "http://www.w3.org/2005/Atom",
    "arxiv": "http://arxiv.org/schemas/atom",
}

records = []

for entry in root.findall("atom:entry", ns):
    title = entry.findtext("atom:title", default=None, namespaces=ns)
    abstract = entry.findtext("atom:summary", default=None, namespaces=ns)

    # arXiv identifier
    arxiv_url = entry.findtext("atom:id", default=None, namespaces=ns)
    arxiv_id = arxiv_url.split("/")[-1] if arxiv_url else None

    authors = []
    affiliations = []

    for author in entry.findall("atom:author", ns):
        name = author.findtext("atom:name", default=None, namespaces=ns)
        affiliation = author.findtext("arxiv:affiliation", default=None, namespaces=ns)

        if name:
            authors.append(name)

        if affiliation:
            affiliations.append(affiliation)

    doi = entry.findtext("arxiv:doi", default=None, namespaces=ns)

    primary_category = None
    primary = entry.find("arxiv:primary_category", ns)
    if primary is not None:
        primary_category = primary.attrib.get("term")

    categories = [
        c.attrib.get("term")
        for c in entry.findall("atom:category", ns)
    ]

    for link in entry.findall("atom:link", ns):
        if link.attrib.get("title") == "pdf":
            pdf_url = link.attrib.get("href")
            break

    records.append({
        "arxiv_id": arxiv_id,
        "doi": doi,
        "title": " ".join(title.split()) if title else None,
        "abstract": " ".join(abstract.split()) if abstract else None,
        "authors": "; ".join(authors) if authors else None,
        "affiliations": "; ".join(dict.fromkeys(affiliations)) if affiliations else None,
        "published": entry.findtext(
                "atom:published",
                default=None,
                namespaces=ns,
            ),
        "primary_category": primary_category,
        "categories": "; ".join(categories),
        "abstract_url": arxiv_url,
        "pdf_url": pdf_url,
    })

df = pd.DataFrame(records)

In [ ]:
output_path = os.path.join("..", "data", f"arxiv_papers.csv")

# df.to_csv(
#     output_path,
#     sep="\t",
#     index=False,
#     encoding="utf-8",
# )

ChemRxiv's primary OpenEngage API can be blocked by Cloudflare (HTTP 403) in some
environments. This module provides a fallback based on Crossref's public API
using the ChemRxiv DOI prefix (``10.26434``).

https://github.com/jannisborn/paperscraper/blob/main/paperscraper/get_dumps/utils/chemrxiv/crossref_api.py

https://connect.acspubs.org/chemrxiv-migration-FAQ

The pdf endpoint is willl not work because of Cloudflare JavaScript challenge

The work around is to start a browser and work it out: too much to do
from pathlib import Path
from playwright.sync_api import sync_playwright


In [ ]:
import requests

query = "polymer"
url = "https://api.crossref.org/works"
params = {
    # "query": query,
    "filter": "prefix:10.26434,type:posted-content",  # 10.26434 = ChemRxiv DOI prefix
    "select": "DOI,title,author,posted,abstract",
}
headers = {"User-Agent": "my-research-script/1.0 (mailto:you@example.com)"}

r = requests.get(url, params=params, headers=headers, timeout=30)
r.raise_for_status()


In [ ]:
r.json()

In [ ]:
papers = []
for item in r.json()["message"]["items"]:
    date_parts = (item.get("posted") or item.get("issued") or {}).get("date-parts", [[]])[0]
    pub_year = str(date_parts[0]) if date_parts else None

    authors = []
    for a in item.get("author", []):
        name = f"{a.get('given', '')} {a.get('family', '')}".strip()
        affs = [af["name"] for af in a.get("affiliation", []) if "name" in af]
        authors.append({"name": name, "affiliations": affs})

    papers.append({
        "doi":      item.get("DOI"),
        "title":    (item.get("title") or [""])[0],
        "abstract": item.get("abstract", ""),   # often empty — see note
        "pub_year": pub_year,
        "authors":  authors,
    })

In [ ]:
import requests

url = "https://chemrxiv.org/doi/pdf/10.26434/chemrxiv.12536153.v2"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/137.0.0.0 Safari/537.36"
    )
}

r = requests.get(url, headers=headers, timeout=60)
print(r.status_code)
print(r.headers.get("content-type"))

In [ ]:
print(r.text[:500])

In [ ]:
import requests

doi = "10.26434/chemrxiv-2023-lw1ns"

r = requests.get(
    f"https://doi.org/{doi}",
    headers={"Accept": "text/html"},
    allow_redirects=True,
)

print(r.url)

In [ ]:
r = requests.get(r.url, timeout=60)
r.raise_for_status()

In [ ]:
import chemrxiv

In [ ]:
client = chemrxiv.Client()

In [ ]:
paper = client.item_by_doi(doi)
paper.download_pdf(dirpath='.', filename='test.pdf')

In [ ]:
load_dotenv()

- 2851294 papers with "query": "polymer* | copolymer*"

In [ ]:
import os
import requests

API_KEY = os.getenv("S2_API_KEY")

headers = {
    "x-api-key": API_KEY
}

FIELDS = ",".join([
    "paperId",
    "title",
    "abstract",
    "year",
    "venue",
    "externalIds",
    "isOpenAccess",
    "openAccessPdf",
])

query = {
    "query": "polymer* | copolymer*",
    "fields": FIELDS,
    "openAccessPdf": "",
    'pdf': True,
}

response = requests.get(
    "http://api.semanticscholar.org/graph/v1/paper/search/bulk",
    headers=headers,
    params=query,
    timeout=60,
).json()

In [ ]:
response

In [ ]:
len(response["data"])

In [ ]:
response["data"][0]

In [ ]:
import os
from pathlib import Path
import requests

EMAIL = "kevin.george@gu.se"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/137.0 Safari/537.36"
    )
}

pdf_dir = Path("pdfs")
pdf_dir.mkdir(exist_ok=True)


def sanitize_filename(name):
    return (
        name.replace("/", "_")
            .replace("\\", "_")
            .replace(":", "_")
            .replace("?", "_")
    )


def try_download(url, outfile):
    try:
        r = requests.get(
            url,
            headers=headers,
            timeout=120,
            allow_redirects=True,
        )

        if (
            r.status_code == 200
            and "pdf" in r.headers.get("content-type", "").lower()
        ):
            outfile.write_bytes(r.content)
            print("Downloaded:", outfile.name)
            return True

        print(
            "failed",
            r.status_code,
            r.headers.get("content-type"),
            url,
        )
        return False

    except Exception as e:
        print("ERROR:", url, e)
        return False


def unpaywall_candidates(doi):
    try:
        r = requests.get(
            f"https://api.unpaywall.org/v2/{doi}",
            params={"email": EMAIL},
            timeout=60,
        )
        r.raise_for_status()

        data = r.json()

        candidates = []

        best = data.get("best_oa_location")
        if best:
            if best.get("url_for_pdf"):
                candidates.append(best["url_for_pdf"])
            if best.get("url"):
                candidates.append(best["url"])

        for loc in data.get("oa_locations", []):
            if loc.get("url_for_pdf"):
                candidates.append(loc["url_for_pdf"])
            if loc.get("url"):
                candidates.append(loc["url"])

        # remove duplicates while preserving order
        seen = set()
        unique = []
        for u in candidates:
            if u not in seen:
                unique.append(u)
                seen.add(u)

        return unique

    except Exception as e:
        print("Unpaywall failed:", doi, e)
        return []


for paper in response["data"]:

    doi = paper.get("externalIds", {}).get("DOI")
    paper_id = paper["paperId"]

    filename = sanitize_filename(doi or paper_id) + ".pdf"
    outfile = pdf_dir / filename

    if outfile.exists():
        continue

    candidates = []

    oa = paper.get("openAccessPdf")
    if oa and oa.get("url"):
        candidates.append(oa["url"])

    if doi:
        candidates.extend(unpaywall_candidates(doi))

    downloaded = False

    for url in candidates:
        if try_download(url, outfile):
            downloaded = True
            break

    if not downloaded:
        print(f"Could not download {doi}")

In [ ]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
}
u = requests.get(
    f"https://onlinelibrary.wiley.com/doi/pdfdirect/10.1002/sstr.202000144",
    headers=headers,
    timeout=60,
)
u.raise_for_status()
data = u.json()

In [ ]:
import requests

doi = "10.1002/sstr.202000144"
email = "kevin.george@gu.se"

u = requests.get(
    f"https://api.unpaywall.org/v2/{doi}",
    params={"email": email},
    timeout=60,
)
u.raise_for_status()
data = u.json()

best = data.get("best_oa_location") or {}
print(best.get("url_for_pdf"))
print(best.get("url"))
print(best.get("host_type"))
print(best.get("license"))

In [ ]:
candidates = []

for loc in [data.get("best_oa_location")] + data.get("oa_locations", []):
    if not loc:
        continue
    if loc.get("url_for_pdf"):
        candidates.append(loc["url_for_pdf"])
    elif loc.get("url"):
        candidates.append(loc["url"])

print(candidates)

In [ ]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
}
def try_download(url, out):
    r = requests.get(
        url,
        headers=headers,
        timeout=120,
        allow_redirects=True,
    )
    if r.status_code == 200 and "pdf" in r.headers.get("content-type", "").lower():
        with open(out, "wb") as f:
            f.write(r.content)
        return True
    print("failed", r.status_code, r.headers.get("content-type"), url)
    return False

for i, url in enumerate(candidates):
    out = f"paper_{i}.pdf"
    if try_download(url, out):
        print("Downloaded:", out)
        break

In [ ]:
import os
import requests

API_KEY = os.getenv("S2_API_KEY")



headers = {
    "x-api-key": API_KEY
}

# 1. Get latest release
releases = requests.get(
    "https://api.semanticscholar.org/datasets/v1/release",
    headers=headers,
    timeout=60,
).json()

latest = releases[-1]
print("Latest release:", latest)

# # 2. Get S2ORC download links
# datasets = requests.get(
#     # f"https://api.semanticscholar.org/datasets/v1/release/{latest}/dataset/s2orc",
#     f"https://api.semanticscholar.org/datasets/v1/release/{latest}",
#     headers=headers,
#     timeout=60,
# ).json()

# files = dataset["files"]
# print("Number of shards:", len(files))

In [ ]:
response = requests.get(
    "https://api.semanticscholar.org/datasets/v1/release",
    headers=headers,
    timeout=60,
)

In [ ]:
response.headers

In [ ]:
response.json()

In [ ]:
datasets = requests.get(
    # f"https://api.semanticscholar.org/datasets/v1/release/{latest}/dataset/s2orc",
    f"https://api.semanticscholar.org/datasets/v1/release/{latest}",
    headers=headers,
    timeout=60,
).json()

In [ ]:
print(datasets["README"])

In [ ]:
for dataset in datasets["datasets"]:
    print(f"Dataset: {dataset['name']}")
    print(f"Description: {dataset['description']}")
    print(dataset["README"])

    print(" ")
    print("--"*100)
    print(" ")

In [ ]:
# 2. Get S2ORC download links
dataset = requests.get(
    f"https://api.semanticscholar.org/datasets/v1/release/{latest}/dataset/papers",
    headers=headers,
    timeout=60,
).json()

files = dataset["files"]
print("Number of shards:", len(files))

In [ ]:
dataset

In [ ]:
files

In [ ]:
from tqdm import tqdm
from urllib.parse import urlparse
from pathlib import Path

In [ ]:
LOCAL_PATH = os.path.join("..", "data", "s2orc_v2")
os.makedirs(LOCAL_PATH, exist_ok=True)

In [ ]:
for url in tqdm(files):
    shard_id = Path(urlparse(url).path).stem
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(os.path.join(LOCAL_PATH, f"{shard_id}.gz"), "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    
print("Downloaded all shards.")